# Course: COMP 333 — Project Phase 1: Data Acquisition & Baseline

**Team D**:
- Ronnie Chan (27206003)
- Patrice Gallant (40301020)
- Nesrine Larbi (40079009)

<br><br>

### **Task Description**
The Phase 1 of this project involves the following steps:
1. Perform Data Acquisition - The uncompressed size of the acquired dataset must be at least 1 GB.
2. Perform Data Wrangling/Cleaning.
3. Perform Data Exploratory Analysis (EDA).
4. Perform a Baseline model on the dataset.

<br><br>

### **Source of the Datasets**
1. NYC Yellow Taxi Trip Data: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
2. Weather Data (API): https://open-meteo.com/en/docs/historical-weather-api

<br><br>

### **Division-of-Labour Statement**
| Student | Tasks |
| :--- | :--- |
| **Ronnie Chan** | 1. Data Acquisition (Section 2) <br> 2. Python scripts: **taxi_data.py**, **taxi_utils.py** <br> 3. Data Wrangling/Cleaning (Sections 3.1, 3.3, 3.4, 3.5, 3.6, 3.7) <br> 4. Baseline Model (Section 5) <br> 5. README.md |
| **Patrice Gallant** | 1. Data Wrangling/Cleaning (Section 3.2) <br> 2. Baseline model (Section 5) |
| **Nesrine Larbi** | 1. Python script: **taxi_utils.py** <br> 2. Data Wrangling/Cleaning (Sections 3.8, 3.9, 3.10) <br> 3. EDA (Section 4) |

## 1. Import Libraries

In [ ]:
# For Open-Meteo API (run once if not installed: pip install openmeteo-requests requests-cache retry-requests)
# %pip install -q openmeteo-requests requests-cache retry-requests

# General Libraries
import pandas as pd
import matplotlib.pyplot as plt
import gc
import numpy as np
import math

# Libraries for Machine Learning
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Python script for DDA and EDA (refactored code)
from taxi_utils import TaxiDDA, TaxiEDA

## 2. Data Acquisition
___

- **Objective**: This section covers the retrieval of our dataset from the [NYC Taxi and Limousine Commission (TLC)](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) website and historical New York weather via the [Open-Meteo API](https://open-meteo.com/en/docs/historical-weather-api).

- **Python Script**: `taxi_data.py`

- **Data Selection:**
  1. **NYC Yellow Taxi Trip Records:** We selected data from **June and July 2025** to represent the Peak Summer Tourism Season 2025.
  2. **NYC Weather Data:** We retrieved hourly weather data for **New York** from **June and July 2025** using the Open-Meteo API.

- **Programmatic Retrieval:** The downloading process is implemented in **`taxi_data.py`** using the `curl` command for the trip records and the Python sample code from the Open-Meteo API documentation. Given the 12.7 GB RAM constraint of our environment (Google Colab), we built our own dataset with a manageable **raw size of 1.4+ GB on disk**.

- **Challenges**: Processing a **1.4+GB dataset** is challenging within a 12.7 GB RAM limit. Since most pandas operations create temporary copies, it is crucial to:
  - **Garbage Collection**: Use `gc.collect()` regularly.
  - **Downcast data types**: Downcast `int32` to `int8` and `float64` to `float32`.
  - **Sampling**: Sample 10K rows for visualization.

In [ ]:
# Perform Data Retrieval — run taxi_data.py to download data files if not already present
# !python taxi_data.py

### 2.1. NYC Yellow Taxi Trip Records (June and July 2025)

First, we merged the downloaded data for June and July 2025 into one dataset `taxi_df`.

In [ ]:
# Read saved datasets into pandas DataFrame
yellow_taxi_path = lambda m: f"yellow_tripdata_2025-{m}.parquet"
june_df = pd.read_parquet(yellow_taxi_path("06"))
july_df = pd.read_parquet(yellow_taxi_path("07"))

# Merge the 2 datasets into a master dataset
taxi_df = pd.concat([june_df, july_df], ignore_index=True)

# Free up memory
del june_df
del july_df
gc.collect()

# Display the first 10 rows
taxi_df.head(10)

### 2.2. NYC Hourly Weather Data

We retrieved hourly weather data for New York City using the Open-Meteo API.

Parameters used:
```python
params = {
  "latitude": 40.7143,
  "longitude": -74.006,
  "start_date": "2025-06-01",
  "end_date": "2025-07-31",
  "hourly": ["temperature_2m", "precipitation"],
  "timezone": "America/New_York",
}
```

In [ ]:
# Read weather data (downloaded via taxi_data.py)
weather_df = pd.read_csv("nyc_weather.csv")
weather_df.head(5)

In [ ]:
# Correct the datetime (remove timezone offset)
weather_df['date'] = pd.to_datetime(weather_df['date']).dt.tz_localize(None)
weather_df.drop(columns=['Unnamed: 0'], inplace=True)

### 2.3. Merge Taxi Data with Weather Data

We performed feature engineering on `taxi_df` to extract specific temporal features (day, hour, day of the week) to enable merging with `weather_df` (hourly weather data).

In [ ]:
# Extract day, hours and day of week from taxi_df
taxi_df['day'] = taxi_df['tpep_pickup_datetime'].dt.day.astype('int8')
taxi_df['pickup_hour'] = taxi_df['tpep_pickup_datetime'].dt.floor('h')
taxi_df['dropoff_hour'] = taxi_df['tpep_dropoff_datetime'].dt.floor('h')
taxi_df['day_of_week'] = taxi_df['tpep_pickup_datetime'].dt.dayofweek.astype('int8')

# Merge the weather_df with the taxi_df
taxi_df = taxi_df.merge(
    weather_df,
    left_on='pickup_hour',
    right_on='date',
    how='left'
)

# Free up memory
taxi_df.drop(columns=['date'], inplace=True)
del yellow_taxi_path
del weather_df
gc.collect()

The following shows the summary of `taxi_df`. The `info()` confirms our merged dataset has the **uncompressed size of 1.4+GB**.

In [ ]:
taxi_df.info(memory_usage=True)

In [ ]:
(taxi_df.memory_usage(deep=True).sum() / 1024**3).item()

### 2.4. NYC Yellow Taxi Trips Data Dictionary
The [Yellow Taxi Trips Data Dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf) describes the meaning of each column:

| Feature Types | Features |
| :--- | :--- |
| **Identifier** | `VendorID`|
| **Temporal** |`tpep_pickup_datetime`, `tpep_dropoff_datetime`|
| **Geographical**| `PULocationID`, `DOLocationID`, `trip_distance`|
| **Categorical**| `RatecodeID`, `payment_type`, `passenger_count` |
| **Money-Value** | `fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge`, `total_amount`, `congestion_surcharge`, `airport_fee`, `cbd_congestion_fee` |
|**Others** | `store_and_fwd_flag` |

In [ ]:
taxi_df.head(5)

In [ ]:
# Save the raw master dataset (uncomment to save)
# taxi_df.to_parquet("taxi_raw.parquet", index=False)

## 3. Data Wrangling
___

The first steps of data wrangling involve the following:
1. Remove unrelevant columns: `store_and_fwd_flag`, `PULocationID`, `DOLocationID`.
2. Inspect the statistical summary of the dataset.

In [ ]:
# 1. Drop unrelevant columns
taxi_df = taxi_df.drop(columns=['store_and_fwd_flag', 'PULocationID', 'DOLocationID'])

# 2. Display statistical summary
dda = TaxiDDA()
dda.quantDDA(taxi_df)

# Free up memory
del dda
gc.collect()

The summary shows that the dataset contains incorrect entries, NaN values, outliers, etc.
___
Thus, the data wrangling/cleaning consists of the following tasks:
1. Handle duplicates and mirrored data
2. Handle negative fare/fee values
3. Handle 0-distance records
4. Handle 0-passenger records
5. Handle NaN values
6. Map numerical codes to descriptive labels
7. Downcast data types
8. Statistical summary and visualization via `quantDDA()` and `vizDDA()`
9. Handle outliers
10. Feature selection

### 3.1. Handle Duplicates

The dataset has 4 duplicates.

In [ ]:
# Compute the number of duplicates
cols = ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_distance', 'total_amount']
num_duplicates = taxi_df.duplicated(subset=cols).sum()
int(num_duplicates)

In [ ]:
# Retrieve the duplicated entries
duplicates = taxi_df[taxi_df.duplicated(subset=cols, keep=False)]

# Sort by pickup time to inspect the duplicates
duplicates.sort_values(by='tpep_pickup_datetime').head(10)

The duplicated rows are almost identical — same `VendorID`, pick up/drop off times. These represent system duplicates. We removed the 4 duplicates.

In [ ]:
# Drop the duplicated row
taxi_df = taxi_df.drop_duplicates(subset=cols, keep='first')

# Free up memory
del num_duplicates
del duplicates
gc.collect()

**Mirrored Entries**

We also found 150,444 mirrored rows in the dataset.

In [ ]:
# Compute the number of mirrored rows
cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_distance']
nb_mirrored = taxi_df.duplicated(subset=cols).sum()
int(nb_mirrored)

In [ ]:
# Retrieve the duplicated/mirrored entries
duplicates = taxi_df[taxi_df.duplicated(subset=cols, keep=False)]

# Sort by pickup time to inspect
duplicates.sort_values(by='tpep_pickup_datetime').head(10)

The **mirrored records** are twins — one with a positive `total_amount` and one with an identical negative value. These represent cancelled transactions. We removed all 150,444 mirrored records.

In [ ]:
# Apply absolute value on the total amount
taxi_df['abs_amount'] = taxi_df['total_amount'].abs()
cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_distance', 'abs_amount']

# Find the mirrored rows (Positive/Negative pairs)
is_mirror = taxi_df.duplicated(subset=cols, keep=False)
taxi_df[is_mirror][['VendorID', 'trip_distance','total_amount','abs_amount']].head(10)

In [ ]:
# Drop mirrored rows
taxi_df = taxi_df[~is_mirror]

### 3.2. Handle Remaining Negative Fare Values

After removing mirrored rows, there are still negative values in financial features. We add an `is_negative` flag to retain these rows without losing data integrity.

In [ ]:
# Check the count of negative values
taxi_df[taxi_df['fare_amount'] < 0].shape[0]

In [ ]:
taxi_df['is_negative'] = (
    (taxi_df['fare_amount'] < 0) | (taxi_df['Airport_fee'] < 0) |
    (taxi_df['cbd_congestion_fee'] < 0) | (taxi_df['tip_amount'] < 0) |
    (taxi_df['total_amount'] < 0) | (taxi_df['congestion_surcharge'] < 0) |
    (taxi_df['improvement_surcharge'] < 0) | (taxi_df['extra'] < 0) |
    (taxi_df['mta_tax'] < 0)
)
taxi_df.head(5)

### 3.3. Handle 0-Distance Rows

The dataset contains 238,440 rows where the taximeter did not record trip distance. We drop these to ensure meaningful predictions.

In [ ]:
# Inspect the number of 0-distance rows
print("Number of 0-distance rows:", len(taxi_df.loc[taxi_df["trip_distance"]==0]))

# Drop the 0-distance rows
taxi_df = taxi_df[taxi_df['trip_distance'] > 0]
print("Successfully dropped")

In [ ]:
# Check: no 0-distance rows remain
0 in taxi_df['trip_distance'].unique()

### 3.4. Handle 0-Passenger Rows

The dataset contains 0-passenger rows (0.54% of dataset). We replace them with the mode (1 passenger). A trip should have at least 1 passenger.

In [ ]:
zero_passenger_df = taxi_df.loc[taxi_df['passenger_count']==0]
print(f"Number of 0-passenger rows: {len(zero_passenger_df)}")
print(f"Ratio of 0-passenger rows: {len(zero_passenger_df)/len(taxi_df)*100: .2f} %")

In [ ]:
# Inspect unique values and mode
print("Unique values:", zero_passenger_df['RatecodeID'].unique())
print("Mode:", zero_passenger_df['RatecodeID'].mode().item())

In [ ]:
# Replace 0 passenger with mode 1
taxi_df.loc[taxi_df['passenger_count'] == 0, 'passenger_count'] = 1
print("Successfully replaced 0 passenger")

In [ ]:
# Check: no 0-passenger rows remain
0 in taxi_df['passenger_count'].unique()

### 3.5. Handle NaN Values

In [ ]:
# Check the number of NaN values in each column
taxi_df.isnull().sum()

**PART 1**

NaN values (~27%) are consistent across 4 features: `passenger_count`, `RatecodeID`, `congestion_surcharge`, `Airport_fee`.

- We impute `congestion_surcharge` and `Airport_fee` with **0** (not travelling to airport / congestion zone).
- We impute `passenger_count` with **mode = 1** (single-passenger trip).
- We fill `RatecodeID` with **99** (Unknown rate).

In [ ]:
# Compute ratio of NaN values
nan_count = taxi_df['congestion_surcharge'].isnull().sum()
nan_ratio = round(nan_count/len(taxi_df) * 100)
print(f"Ratio of NaN values: {nan_ratio} %")

In [ ]:
print('Airport_fee')
print('---------------------')
print('Unique values:', taxi_df['Airport_fee'].unique())
print('Average', taxi_df.loc[taxi_df['Airport_fee'] > 0, 'Airport_fee'].mean().round(2))

print('\n\nCongestion Surcharge')
print('---------------------')
print('Unique values:', taxi_df['congestion_surcharge'].unique())
print('Average', taxi_df.loc[taxi_df['congestion_surcharge'] > 0, 'congestion_surcharge'].mean().round(2))

In [ ]:
# Replace NaNs with 0
taxi_df.loc[:, ['congestion_surcharge', 'Airport_fee']] = taxi_df[['congestion_surcharge', 'Airport_fee']].fillna(0)

In [ ]:
# Show the mode
print('Mode for passenger_count:', taxi_df['passenger_count'].mode().item())

# Replace NaNs with mode 1
taxi_df.loc[:, ['passenger_count']] = taxi_df[['passenger_count']].fillna(1)

# Replace NaNs with code 99 (Unknown)
taxi_df.loc[:, ['RatecodeID']] = taxi_df[['RatecodeID']].fillna(99)

**PART 2 — WEATHER**

Rows with NaN weather values fall outside our analysis period (2025-06-01 to 2025-07-31):
- 1 record from 2009
- 20 records from 2025-05-31

We drop these 21 records.

In [ ]:
# Inspect records with NaN values for hourly weather
nan_weather_df = taxi_df.loc[taxi_df['temperature_2m'].isnull()]
nan_weather_df

In [ ]:
# Verify same indexes
taxi_df.loc[taxi_df['pickup_hour'] < '2025-06-01 00:00:00'].index.equals(nan_weather_df.index)

In [ ]:
# Drop out-of-period records
taxi_df = taxi_df.drop(nan_weather_df.index)
del cols, duplicates, nb_mirrored, is_mirror
del zero_passenger_df
del nan_weather_df
del nan_count, nan_ratio
gc.collect()

In [ ]:
# Check no invalid entries after July 2025
taxi_df.loc[taxi_df['pickup_hour'] > '2025-07-31 23:59:59']

**Final Check:** No missing values remain.

In [ ]:
taxi_df.isnull().sum()

### 3.6. Map Labels

Converting numerical codes to descriptive labels using the [NYC Yellow Taxi Trips Data Dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf).

#### 3.6.1 Feature: `RatecodeID`

In [ ]:
taxi_df['RatecodeID'].unique()

In [ ]:
# Define the rate code dictionary
rate_code_map = {
    1: "Standard rate",
    2: "JFK",
    3: "Newark",
    4: "Nassau or Westchester",
    5: "Negotiated fare",
    6: "Group ride",
    99: "Unknown"
}

# Map the rate code ID with the appropriate category
taxi_df.loc[:,'rate_code'] = taxi_df['RatecodeID'].map(rate_code_map).astype('category')

# Free up memory
taxi_df = taxi_df.drop(columns=['RatecodeID'])
gc.collect()

In [ ]:
# Check: rate codes correctly mapped
taxi_df['rate_code'].unique()

#### 3.6.2 Feature: `payment_type`

In [ ]:
taxi_df['payment_type'].unique()

In [ ]:
# Define the payment dictionary
payment_map = {
    0: "Flex Fare",
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip"
}

# Apply mapping and categorize
taxi_df.loc[:,'payment_type'] = taxi_df['payment_type'].map(payment_map).astype('category')

In [ ]:
# Check: payment types correctly mapped
taxi_df['payment_type'].unique()

### 3.7. Downcast dtypes

Reduce numerical dtype to smaller types to efficiently manage memory usage.

In [ ]:
# Drop unrelevant columns
taxi_df = taxi_df.drop(columns=['VendorID','abs_amount'])

In [ ]:
# Define the numerical columns to convert
float64_cols = [
    'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount',
    'tolls_amount', 'improvement_surcharge', 'total_amount',
    'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee',
    'temperature_2m', 'precipitation'
]

# Convert to a smaller dtype (downcast)
taxi_df['passenger_count'] = taxi_df['passenger_count'].astype('int8')
taxi_df[float64_cols] = taxi_df[float64_cols].astype('float32')

# Delete temporary variables
del rate_code_map
del payment_map
del float64_cols
gc.collect()

In [ ]:
taxi_df.info()

In [ ]:
taxi_df.head(5)

### 3.8. Initial DDA: `quantDDA()` and `vizDDA()`

Using `TaxiDDA()` from `taxi_utils.py` to generate statistical summaries and visualizations.

#### 3.8.1. Part 1: `quantDDA()`

In [ ]:
dda = TaxiDDA()
dda.quantDDA(taxi_df)

**Interpretation of Statistical Summary:**
- The dataset still contains negative values for money-value features (see `min`) — intentional, flagged by `is_negative`.
- The dataset contains outliers — handled in **Section 3.9**.
- The dataset contains redundant columns — handled in **Section 3.10**.

#### 3.8.2. Part 2: `vizDDA()`

We execute `vizDDA()` on a **smaller sample of 10K entries** using 8 selected columns to avoid overcrowded visualization.

In [ ]:
# Randomly sample a batch of 10K entries
sample_df = taxi_df.sample(n=10000, random_state=42)
sample_df = sample_df.loc[~sample_df["is_negative"]]
cols = ["trip_distance", "total_amount", "tip_amount", "passenger_count",
        "payment_type", "rate_code", "day_of_week", "temperature_2m"]
dda.vizDDA(sample_df[cols])

**Interpretation:**
- The heatmap confirms no more NaN values.
- `tip_amount` is higher when the passenger travels alone, pays by credit card, and is going to the airport.
- Greater `trip_distance` is positively correlated with a larger `tip_amount`.

In [ ]:
# Free up memory
del sample_df, cols, dda
gc.collect()

### 3.9. Handle Outliers

**Strategy: cap at the 99th percentile (Winsorization)**

We chose *capping* over *dropping* to:
- Preserve the maximum number of rows for model training
- Avoid removing legitimate edge cases (e.g., long airport trips)
- Reduce the statistical influence of erroneous extremes

| Feature | Cap applied |
|---|---|
| `trip_distance` | 99th percentile |
| `fare_amount` | 99th percentile |
| `total_amount` | 99th percentile |
| `tip_amount` | 99th percentile |
| `tolls_amount` | 99th percentile |

In [ ]:
outlier_cols = ['trip_distance', 'fare_amount', 'total_amount', 'tip_amount', 'tolls_amount']

# Compute 99th percentile caps
caps = taxi_df[outlier_cols].quantile(0.99)
print("99th percentile caps:\n", caps.round(2), "\n")

# Count outliers above cap before capping
before = (taxi_df[outlier_cols] > caps).sum()
print("Rows above cap (before):\n", before, "\n")

# Cap at 99th percentile
for col in outlier_cols:
    taxi_df[col] = taxi_df[col].clip(upper=caps[col])

# Verify — no values above cap remain
after = (taxi_df[outlier_cols] > caps).sum()
print("Rows above cap (after):\n", after)
print(f"\nDataset shape after capping: {taxi_df.shape[0]:,} rows × {taxi_df.shape[1]} columns")

# Free up memory
del caps, before, after
gc.collect()

### 3.10. Feature Selection

We select features for the Machine Learning models. Raw datetime columns are dropped since temporal information is captured in `day`, `day_of_week`, and `hour_of_day`.

**Features kept (14 columns):**

| Feature | Type | Role |
|---|---|---|
| `trip_distance` | float32 | Core trip metric |
| `total_amount` | float32 | Target (RQ1) / cluster feature (RQ2) |
| `tip_amount` | float32 | Behavioral feature |
| `fare_amount` | float32 | Base fare |
| `congestion_surcharge` | float32 | Fee feature |
| `Airport_fee` | float32 | Fee feature |
| `passenger_count` | int8 | Trip attribute |
| `rate_code` | category | Trip category |
| `payment_type` | category | Trip category |
| `day_of_week` | int8 | Temporal feature |
| `day` | int8 | Temporal feature |
| `hour_of_day` | int8 | Temporal feature |
| `temperature_2m` | float32 | Weather feature |
| `precipitation` | float32 | Weather feature |

**Dropped:** `tpep_pickup_datetime`, `tpep_dropoff_datetime`, `pickup_hour`, `dropoff_hour`

> **Note for RQ1:** fee subcomponents (`extra`, `mta_tax`, `improvement_surcharge`, `cbd_congestion_fee`, `tolls_amount`, `fare_amount`, `tip_amount`) are components of `total_amount` and will be excluded at model training to avoid data leakage.

In [ ]:
# Extract hour of day as integer before dropping pickup_hour
taxi_df['hour_of_day'] = taxi_df['pickup_hour'].dt.hour.astype('int8')

# Drop raw datetime columns
cols_to_drop = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'dropoff_hour',
]
taxi_df = taxi_df.drop(columns=cols_to_drop)

# Free up memory
del cols_to_drop
gc.collect()

print(f"Final shape: {taxi_df.shape[0]:,} rows × {taxi_df.shape[1]} columns")
taxi_df.dtypes

In [ ]:
# Save the final feature-selected dataset
# taxi_df.to_parquet('taxi_cleaned.parquet', index=False)
# print("Saved to taxi_cleaned.parquet")

## 4. Exploratory Data Analysis (EDA)
___

### 4.1. Summary Statistics

The following shows the summary statistics of the entire dataset.

In [ ]:
eda = TaxiEDA()
eda.summary_stats(taxi_df)

### 4.2. Univariate Visualizations

The following plots show the individual distribution of each key feature:
1. **Trip Distance** — right-skewed; most trips are short (under 5 miles)
2. **Total Amount** — right-skewed; majority of fares are under $40
3. **Tip Amount** — heavily right-skewed; many trips have $0 tip (cash payments)
4. **Passenger Count** — majority are single-passenger trips
5. **Payment Type** — credit card is the dominant payment method
6. **Rate Code** — standard rate (1) dominates; a small proportion are airport trips
7. **Day of Week** — relatively uniform; slight dip on Sundays
8. **Temperature** — bimodal reflecting June/July seasonal variation in NYC

**NOTE:** Visualization is executed on a representative random sample of 10K entries.

In [ ]:
eda.plot_univariate(taxi_df)

### 4.3. Bivariate Visualizations

The following plots explore relationships between pairs of features:
1. **Trip Distance vs Total Amount** — expected positive linear relationship
2. **Total Amount by Payment Type** — how payment method relates to fare value
3. **Total Amount by Rate Code** — fare differences across rate categories (e.g., airport trips)
4. **Trip Volume by Hour of Day** — demand patterns across the day
5. **Trip Distance vs Tip Amount** — longer trips attract higher tips
6. **Temperature vs Average Total Amount** — weather conditions affect fare

**NOTE:** Visualization is executed on a representative random sample of 10K entries.

In [ ]:
eda.plot_bivariate(taxi_df)

### 4.4. Correlation Analysis

The heatmap below shows pairwise Pearson correlations between all numerical features.

In [ ]:
eda.plot_correlation(taxi_df)

### 4.5. Research Questions

Based on the exploratory analysis above, we formulate the following two research questions:

---

**RQ1 — Supervised Learning:**
> *"Can we classify a high-tipper passenger based on trip-selected features (distance, time of day, day of week, rate code, passenger count) and weather conditions (temperature, precipitation)?"*

- **Type:** Classification
- **Models:** XGBoost and Random Forest
- **Target variable:** `is_high_tip` (Boolean: 1 if tip > 20% of the fare amount, 0 otherwise)
- **Features:** `trip_distance`, `pickup_hour` (as hour-of-day), `day_of_week`, `passenger_count`, `temperature_2m`, `precipitation`

---

**RQ2 — Unsupervised Learning:**
> *"Can we identify the distinct natural clusters of NYC taxi trips based on their temporal and financial features to study customer behavior?"*

- **Type:** Clustering (K-Means)
- **Features:** `trip_distance`, `total_amount`, `tip_amount`, `pickup_hour` (as hour-of-day), `day_of_week`, `passenger_count`, `temperature_2m`, `precipitation`

## 5. Baseline Model
___

We compare three Linear Regression models for target feature `fare_amount`:
- **Model A**: 1 input — `trip_distance`
- **Model B**: 2 inputs — `trip_distance` and `tip_amount`
- **Model C**: 3 inputs — `trip_distance`, `tip_amount`, and `Airport_fee`

We chose `trip_distance` as our main predictor (Pearson r = 0.87 with `fare_amount`). `airport_fee` and `tip_amount` were chosen for their non-negligible Pearson correlations (0.56 and 0.54).

Before training, we exclude rows with negative fares (refunds/adjustments) that would skew the linear regression model.

In [ ]:
# Create a sample dataframe: 10,000 rows where is_negative is False
regression_sample_df = taxi_df[taxi_df['is_negative'] == False].sample(n=10000, random_state=1)

# Create numpy arrays for our feature X and target Y
X = regression_sample_df[['trip_distance', 'tip_amount', 'Airport_fee']].to_numpy()
Y = regression_sample_df['fare_amount'].to_numpy()

### 5.1. Splitting Phase

We use `train_test_split` twice for a 70%/15%/15% split (training/validation/test).

In [ ]:
# 1. Get the 15% Testing set
x_temp, x_test, y_temp, y_test = train_test_split(X, Y, test_size=0.15, random_state=1)

# 2. Get 15% Validation set from the remaining 85%
valid_ratio = 0.15 / 0.85
x_train, x_valid, y_train, y_valid = train_test_split(x_temp, y_temp, test_size=valid_ratio, random_state=1)

# Check splits
print(f"Train set: {len(x_train)/len(X):.1%}")
print(f"Valid set: {len(x_valid)/len(X):.1%}")
print(f"Test set:  {len(x_test)/len(X):.1%}")

### 5.2. Training Phase

In [ ]:
# Model A: trip_distance only
modelA = LinearRegression()
x_train_A = x_train[:, 0][:, np.newaxis]
modelA.fit(x_train_A, y_train)

# Model B: trip_distance and tip_amount
modelB = LinearRegression()
modelB.fit(x_train[:,:2], y_train)

# Model C: all features
modelC = LinearRegression()
modelC.fit(x_train, y_train)

### 5.3. Validation Phase

**Performance metrics:** MSE (lower is better) and R² (higher is better).

In [ ]:
# Get predictions on the validation set
x_valid_A = x_valid[:, 0][:, np.newaxis]
predictA = modelA.predict(x_valid_A)
predictB = modelB.predict(x_valid[:,:2])
predictC = modelC.predict(x_valid)

# Performance metrics for Model A
print(f"MSE for model A: {mean_squared_error(y_valid, predictA):.4f}")
print(f"R2 for model A: {modelA.score(x_valid_A, y_valid):.4f}")

# Performance metrics for Model B
print(f"\nMSE for model B: {mean_squared_error(y_valid, predictB):.4f}")
print(f"R2 for model B: {modelB.score(x_valid[:,:2], y_valid):.4f}")

# Performance metrics for Model C
print(f"\nMSE for model C: {mean_squared_error(y_valid, predictC):.4f}")
print(f"R2 for model C: {modelC.score(x_valid, y_valid):.4f}")

**Model Performance:**
- All three models give similar MSE and R², with minor differences.
- **Model B** has the lowest MSE and highest R².
- **Model Selection:** We selected **Model B** as our baseline model for Phase 1.
  1. It has the lowest MSE and a higher R².
  2. It provides higher interpretability than Model C, while being more complex than Model A.

### 5.4. Testing Phase

In [ ]:
test_predict = modelB.predict(x_test[:, :2])
print(f"MSE on test for model B: {mean_squared_error(y_test, test_predict):.4f}")
print(f"R2 on test for model B: {modelB.score(x_test[:, :2], y_test):.4f}")

We visualize the model using a scatter plot and regression line: $y = a_1x_1 + a_2x_2 + b$

In [ ]:
# Extract parameters from baseline model B
b  = modelB.intercept_.item()
a1 = modelB.coef_[0].item()
a2 = modelB.coef_[1].item()

print(f"Bias: {b:0.3f}")
print(f"Coefficient a1: {a1:0.3f}")
print(f"Coefficient a2: {a2:0.3f}")
print(f"\nEquation of Linear Regression Model B:")
print(f"y = {a1:0.3f} x1 + {a2:0.3f} x2 + {b:0.3f}")

In [ ]:
# Define the trend line equation for Model B
y_pred_test = lambda x1, x2: a1 * x1 + a2 * x2 + b

# Plot the Regression Line for Model B
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Linear Regression for Model B', fontsize=18, y=1.01)

# Fare Amount vs Trip Distance (hold x2 = 0)
x1_test = x_test[:,0]
ax[0].scatter(x1_test, y_test, alpha=0.3)
ax[0].plot(x1_test, y_pred_test(x1_test, 0), color="red",
           label=f'$y = {a1:0.3f}x_1 + {b:0.3f}$')
ax[0].set_xlabel('Trip Distance (miles)')
ax[0].set_ylabel('Fare Amount ($)')
ax[0].set_title('Fare Amount vs Trip Distance (Holding $x_2=0$)')
ax[0].legend(loc='upper left')

# Fare Amount vs Tip Amount (hold x1 = 0)
x2_test = x_test[:,1]
ax[1].scatter(x2_test, y_test, color='orange', alpha=0.3)
ax[1].plot(x2_test, y_pred_test(0, x2_test), color="red",
           label=f'$y = {a2:0.3f}x_2 + {b:0.3f}$')
ax[1].set_xlabel('Tip Amount ($)')
ax[1].set_ylabel('Fare Amount ($)')
ax[1].set_title('Fare Amount vs Tip Amount (Holding $x_1=0$)')
ax[1].legend(loc='upper left')

plt.tight_layout()
plt.show()

**Interpretation:**
- `trip_distance` and `fare_amount` are highly correlated.
- `tip_amount` shows significantly higher variance — a **weaker predictor** than `trip_distance`.
- The large number of 0-values in `tip_amount` causes heteroscedasticity.
- Model B is a useful baseline: it confirms trip distance has a measurable impact on fare amount.

<br>

## 6. References
___

1. New York City Taxi and Limousine Commission (TLC). (2025). TLC Trip Record Data. https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page [Accessed: Feb. 26 2026]
2. New York City Taxi and Limousine Commission (TLC). (2025). Data Dictionary – Yellow Taxi Trip Records. https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf [Accessed: Feb. 26 2026]
3. Open-Meteo. (2025). Historical Weather API (New York City: June–July 2025). https://open-meteo.com/en/docs/historical-weather-api [Accessed: Feb. 26 2026]